# Getting started with MemsArrayWS objects

The `MemsArrayWS` class allows getting signals from a remote antenna running a local *Megamicros Broadcast Server (MBS)* server. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.ws import MemsArrayWS

log.setLevel( "INFO" )

# Set server access credentials
#HOST = 'buzenval20.fr'
HOST = 'parisparc.biimea.tech'
#HOST = 'localhost'
PORT = 9002

## Connecting to the remote server

Providing a *MBS* server is running at ``HOST:PORT``, one can try to connect by creating a ``MemsArrayWS`` object.

In [ ]:
# Define the antenna
try:
    antenna = MemsArrayWS( HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )


2023-11-23 10:05:07,435 [INFO]:  .Install MemsArrayWS settings
2023-11-23 10:05:07,450 [INFO]:  .Created a new antenna
2023-11-23 10:05:07,451 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-11-23 10:05:07,456 [INFO]:  .Try connecting to ws://parisparc.biimea.tech:9002...


2023-11-23 10:05:07,718 [INFO]:  .Received positive answer from server
2023-11-23 10:05:07,719 [INFO]:  .Getting settings values from remote receiver...


## Getting remote settings

By using the `settings` command, you can get the settings of the remote antenna:

In [ ]:
# Perform an antenna settings
antenna.settings()

# Print some results
print( f"Available mems: {antenna.available_mems}" )
print( f"Available analogs: {antenna.available_analogs}" )
print( f"Default sampling freequency: {antenna.sampling_frequency} Hz" )

Available mems: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
Available analogs: []
Default sampling freequency: 50000 Hz


2023-11-05 23:54:15,766 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-05 23:54:15,769 [INFO]:  .Connected
2023-11-05 23:54:15,770 [INFO]:  .Send settings command to server
2023-11-05 23:54:15,772 [INFO]:  .Remote server settings command successfull
2023-11-05 23:54:15,774 [INFO]:  .Install MemsArray settings
2023-11-05 23:54:15,775 [INFO]:  .Set datatype to int32 
2023-11-05 23:54:15,775 [INFO]:  .New settings:
2023-11-05 23:54:15,776 [INFO]:   > available_mems: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
2023-11-05 23:54:15,777 [INFO]:   > available_analogs: []
2023-11-05 23:54:15,778 [INFO]:   > datatype: int32
2023-11-05 23:54:15,778 [INFO]:   > frame_length: 512
2023-11-05 23:54:15,780 [INFO]:   > mems_sensibility: 7.539965556205927e-06
2023-11-05 23:54:15,780 [INFO]:   > sampling_frequency: 50000.0 Hz
2023-11-05 23:54:15,781 [INFO]:   > system_type: Mu32


## Performing a selftest

It may be a good idea to run a *selftest* of the remote antenna before getting settings.
Indeed, the antenna could have been chanded or modified since the last request. 
IOn addition, when performing a `selftest()`, new settings values are sent as response of the request so that you do not have to send a new `settings()` request.

In [ ]:
# Perform an antenna selftest
antenna.selftest()

# getting some antenna settings
print( f"Available mems: {antenna.available_mems}" )
print( f"Available analogs: {antenna.available_analogs}" )
print( f"Default sampling freequency: {antenna.sampling_frequency} Hz" )

## Halting the remote server
Notice that halting the server causes the connection to be lost. 

In [ ]:
# Stop the remote server
antenna.shutdown()


## Running

### Getting signals from some MEMs

In [ ]:
# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [1, 2],
    duration=2,
    frame_length=512,
    signal_q_size = 0,
)

# Init a np.ndarray
signals = np.ndarray( (0, antenna.channels_number ) )

# Get signals
i = 0
for data in antenna:
    i += 1
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
antenna.wait()
print( f"Exit from loop. Received {i} frames. Signal shape is: {np.shape( signals )}" )

## Plotting signals

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/antenna.sampling_frequency
fig, axs = plt.subplots( antenna.channels_number )
fig.suptitle('Channels activity')	
for s in range( antenna.channels_number ):
    axs[s].plot( time, signals[:,s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()

## Saving signals as wav file
Since wavfiles are audio files, you cannot save more than 2 channels.

In [ ]:
import wave

WAV_FILENAME = 'toto.wav'

# 2 seconds run, getting signals from MEMs 1 and 2
antenna.run(
    mems = [1, 2],
    duration=10,
    frame_length=512,
    signal_q_size = 0,
)

with  wave.open( WAV_FILENAME, mode='wb' ) as wavfile:
    wavfile.setnchannels(2)
    wavfile.setsampwidth(2)
    wavfile.setframerate( antenna.sampling_frequency )

    # Get signals
    for data in antenna:
        signal = data >> 4
        wavfile.writeframesraw( np.int16( np.reshape( signal, np.size( signal ), order='F' ) ) )

# waiting for the end of the running thread is mandatory
antenna.wait()

## Hearing signal with *pyaudio* library

Note that `signal_q_size` is set to 0, setting the internal queue to an infinite length.
This prevents breaks in the audio stream.

In [6]:
import pyaudio

FRAME_LENGTH = 512
SAMPLING_FREQUENCY = 50000
antenna.setSamplingFrequency( SAMPLING_FREQUENCY )

# Instantiate PyAudio and initialize PortAudio system resources (1)
p = pyaudio.PyAudio()

# Open stream
stream = p.open(
    format = pyaudio.paFloat32,
    channels = 2,
    rate = int( antenna.sampling_frequency ),
    output=True,
    frames_per_buffer=FRAME_LENGTH,
)

# Start running the remote Megamicros system
antenna.run( 
    mems=[17, 18],
    duration=10,
    frame_length=FRAME_LENGTH,
    counter_skip = True,
    signal_q_size = 0
)

# Get signals
transfers_counter = 0
for data in antenna:
    signal = data >> 4

    # convert into float and normalize with MEMs sensibility
    data = ( data.astype( np.float32 ) * antenna.sensibility )

    # write into audio stream
    stream.write( data, num_frames=FRAME_LENGTH )
    transfers_counter += 1

# Close stream and release PortAudio system resources (5)
stream.close()            
p.terminate()

antenna.wait()


2023-11-05 23:55:59,358 [INFO]:  .Starting run execution
2023-11-05 23:55:59,366 [INFO]:  .Install MemsArray settings


MuWSException: Cannot run: settings laoding failed (MuException): Run failed on settings: Some activated microphones ([17 18]) are not available on antenna.

## Saving signals as H5 files

You can save signal in H5 file format. In this example sigansl are saved on the MBS remote server.
The antenna receive no more signals. 

In [ ]:
antenna.run(
    mems = [3, 4],
    duration=2,
    frame_length=512,
    h5_recording=True,                          # H5 recording ON
    h5_pass_through=True,                       # perform F5 recording on server
    h5_rootdir='./',                            # directory where to save file
    h5_compressing=False,                       # Use compression or not
    background_mode=True,
    signal_q_size = 0,
)

antenna.wait()

## Getting signals yourself

In this example, signals are received using the antenna internal queue.

In [ ]:
import queue

antenna.run(
    mems = [1, 2],
    duration=2,
    frame_length=512,
    signal_q_size = 0,
)

i = 0
while True:
    try:
        data = antenna.signal_q.get( timeout=5 )
        print( f"[{i}]" )
        i += 1
        # do what you want with data...

    except queue.Empty:
        print( f"exit from loop at i={i}" )
        break

antenna.wait()

## Listening to the Megamicro remote server
By starting a *master* run on the server, you can connect to the server from others hosts and listening to the signal stream.

### Staring the master run
This call lets the remote server starting a run in the background mode.

In [ ]:

antenna.run(
    mems = [0, 1, 2, 3, 4, 5, 6, 7],
    duration=0,
    frame_length=512,
    signal_q_size=0,
    job='master', 
)

In [ ]:
# Define the antenna
try:
    listener = MemsArrayWS( HOST, port=PORT )
except Exception as e:
    print( f"Failed: {e}" )


Run a listener job. 
Please make attention to always specified the `job` value. Otherwise your next run will start with the old `job` value. 

Beware that if you start a listening job without a limited duration, then you have no way to stop it unless you create your own thread for that... 

In [ ]:
listener.setFrameLength(512)
listener.run(
    mems = [0, 1],
    frame_length=512,
    signal_q_size=0,
    duration=10,
    job='listen'
)

# Init a np.ndarray
signals = np.ndarray( (0, listener.channels_number ) )

# Get signals
i = 0
for data in listener:
    i += 1
    signals = np.concatenate( ( signals, data ), axis=0 )

# waiting for the end of the running thread is mandatory
listener.wait()
print( f"Exit from loop. Received {i} frames. Signal shape is: {np.shape( signals )}" )

In [ ]:
# plot signals
time = np.array( range( np.size(signals,0) ) )/listener.sampling_frequency
fig, axs = plt.subplots( listener.channels_number )
fig.suptitle('Channels activity')	
for s in range( listener.channels_number ):
    axs[s].plot( time, signals[:,s] )
    axs[s].set( xlabel='time in seconds', ylabel='channel %d' % s )

plt.show()

In [ ]:
# halt job
antenna.halt()

In [ ]:
# halt server
antenna.shutdown()

2023-11-05 23:56:41,282 [INFO]:  .Async event loop already running. Adding coroutine to the event loop...


2023-11-05 23:56:41,286 [INFO]:  .Connecting to remote host localhost:9002...
2023-11-05 23:56:41,289 [INFO]:  .Connected
2023-11-05 23:56:41,290 [INFO]:  .Send shutdown command to server
2023-11-05 23:56:41,292 [INFO]:  .Remote server shutdown success
